# ⚡ Swizzle Worker — Colab Pro GPU Inference Node (NVIDIA A100 / L4)
This notebook runs **vLLM** with **distributed prefix caching** on Colab Pro's A100/L4 GPU and exposes an instant, secure public HTTPS tunnel via **Cloudflare Tunnels (`cloudflared`)** to connect directly to your Swizzle Gateway.

In [ ]:
# 1. Verify GPU Allocation (Expect NVIDIA A100-SXM4-40GB/80GB or L4)
!nvidia-smi

In [ ]:
# 2. Install vLLM and Cloudflared Tunnel Binary
!pip install -q vllm
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ vLLM and cloudflared installed successfully!")

In [ ]:
# 3. Start vLLM Model Server in Background with Prefix Caching Enabled
import subprocess, time, socket

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

print(f"🚀 Spawning vLLM worker on port 8000 with model {MODEL_ID}...")
vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--enable-prefix-caching",
    "--max-model-len", "16384",
    "--gpu-memory-utilization", "0.92",
    "--trust-remote-code",
    "--port", "8000"
]

vllm_process = subprocess.Popen(vllm_cmd)

# Wait until port 8000 is open
for _ in range(60):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1)
        s.connect(("127.0.0.1", 8000))
        s.close()
        print("\n✅ vLLM OpenAI API Server is READY on port 8000!")
        break
    except:
        time.sleep(2)
        print(".", end="", flush=True)

In [ ]:
# 4. Launch Cloudflare Tunnel to get your Public HTTPS Worker URL
import re

tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://localhost:8000"]
tunnel_process = subprocess.Popen(tunnel_cmd, stderr=subprocess.PIPE, universal_newlines=True)

public_url = None
for line in tunnel_process.stderr:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print("\n" + "="*70)
        print(f"⚡ SWIZZLE UPSTREAM WORKER LIVE URL: {public_url}")
        print("="*70)
        print("\nCopy this URL and run your Swizzle gateway with:")
        print(f"$env:UPSTREAM_URL = \"{public_url}/v1\"")
        print(".\\swizzle-gateway.exe")
        break

# Keep cell alive to maintain worker tunnel
tunnel_process.wait()